# Análisis de desempeño de plantas de tratamiento — AquaLimpia S. A.

**Proyecto de Ciencia de Datos — Semana 8**

**Objetivo:** analizar el comportamiento de las plantas de tratamiento de aguas
residuales de AquaLimpia S. A., identificar patrones asociados a los
incumplimientos intermitentes en la DBO del efluente y generar información
diferenciada para las áreas de **Operaciones** y **Gestión Ambiental**.

**Datos:** `dataset_set_A_aguas_residuales.xlsx` (200 registros, 3 plantas,
periodo 2025-07-01 a 2025-10-28).

**Flujo de trabajo:** carga → evaluación de calidad de datos → cálculo de
indicadores (NumPy/SciPy) → dashboard exploratorio → exportación de reportes
por área → persistencia de resultados (Joblib).

In [ ]:
import sys
sys.path.append("../src")

from funciones_aguas import (
    cargar_datos,
    evaluar_calidad_datos,
    calcular_eficiencia_remocion,
    resumen_por_planta,
    intervalo_confianza_DBO_salida,
    prueba_anova_plantas,
    exportar_reporte_operaciones,
    exportar_reporte_ambiental,
    guardar_indicadores,
)
import matplotlib.pyplot as plt
import numpy as np

## 1. Carga de datos

In [ ]:
df = cargar_datos("../data/dataset_set_A_aguas_residuales.xlsx")
print(df.shape)
df.head()

## 2. Evaluación de calidad de datos

In [ ]:
COLUMNAS_NUMERICAS = [
    "caudal_entrada_m3_d", "DBO_entrada_mg_L", "SST_entrada_mg_L",
    "pH_entrada", "energia_aeracion_kWh", "lodos_generados_kg_d", "DBO_salida_mg_L",
]
calidad = evaluar_calidad_datos(df, COLUMNAS_NUMERICAS)
calidad

## 3. Cálculo de indicadores de desempeño

In [ ]:
df = calcular_eficiencia_remocion(df)
resumen_plantas = resumen_por_planta(df)
resumen_plantas

### 3.1 Intervalos de confianza (95%) para la DBO de salida por planta (SciPy)

In [ ]:
ic_dbo = intervalo_confianza_DBO_salida(df)
ic_dbo

### 3.2 Prueba ANOVA: ¿existen diferencias significativas entre plantas?

In [ ]:
anova = prueba_anova_plantas(df)
anova

**Interpretación:** un p-valor superior a 0.05 indica que no existe evidencia
estadística suficiente para afirmar que la DBO de salida difiere entre plantas.
Esto respalda la hipótesis de la Ing. Rojas: los incumplimientos son
intermitentes y no responden a un patrón asociado a una planta específica,
sino probablemente a variaciones puntuales en el caudal o la carga de entrada.

## 4. Dashboard exploratorio

In [ ]:
import matplotlib.image as mpimg
%matplotlib inline

# El dashboard se genera con el script analisis_principal.py
# (ver ../outputs/dashboards/dashboard_aqualimpia.png)
img = mpimg.imread("../outputs/dashboards/dashboard_aqualimpia.png")
plt.figure(figsize=(14,10))
plt.imshow(img)
plt.axis("off")
plt.show()

## 5. Exportación de reportes diferenciados por área

In [ ]:
exportar_reporte_operaciones(df, "../outputs/reportes/reporte_operaciones.xlsx")
exportar_reporte_ambiental(df, "../outputs/reportes/reporte_gestion_ambiental.xlsx")
print("Reportes generados correctamente.")

## 6. Persistencia de indicadores (Joblib)

In [ ]:
indicadores = {
    "calidad_datos": calidad,
    "resumen_por_planta": resumen_plantas.to_dict(),
    "intervalos_confianza_DBO_salida": ic_dbo,
    "anova_DBO_salida": anova,
}
guardar_indicadores(indicadores, "../outputs/indicadores_aqualimpia.joblib")
print("Indicadores guardados para su reutilización en futuros periodos de análisis.")

## 7. Conclusiones del análisis exploratorio

- Los incumplimientos normativos son transversales a las tres plantas
  (tasas de incumplimiento entre 70% y 83%), y la prueba ANOVA no muestra
  diferencias estadísticamente significativas entre ellas.
- La eficiencia promedio de remoción de DBO es similar entre plantas
  (~87%), lo que sugiere que el proceso de tratamiento en sí funciona de
  forma comparable, pero no logra estabilizar el efluente por debajo del
  umbral normativo de forma consistente.
- La evolución temporal de la DBO de salida muestra oscilaciones
  irregulares alrededor del límite de referencia (35 mg/L), consistente
  con la ausencia de un patrón definido reportada por el área de
  operaciones.
- Se recomienda profundizar el análisis incorporando la relación entre
  caudal de entrada / carga contaminante y los eventos de incumplimiento,
  como línea de trabajo futura.